# Dota 2: дополнительные признаки

В этом ноутбуке я проверяю, что можно добавить поверх baseline: командный чат, TF-IDF и статистики раннего преимущества по золоту/опыту. В исходной домашке было больше идей, но здесь оставлены только реально проработанные эксперименты.


## 1. Setup

Подключаю baseline pipeline и инструменты для text preprocessing, sparse matrices, scaling и логистической регрессии.


In [ ]:
from sklearn.linear_model import LogisticRegression
import sys
from pathlib import Path
sys.path.extend([str(Path.cwd() / "src"), str(Path.cwd().parent / "src")])
from dota_ml.pipeline import run_base_pipeline
import pandas as pd
import re
from functools import lru_cache
import nltk
from nltk.tokenize import wordpunct_tokenize
from nltk.corpus import stopwords
import pymorphy3
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import StandardScaler
from dota_ml.pipeline import (
    build_hero_matrices,
    fit_region_mmr_artifacts,
    transform_region_mmr,
)


def gini(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1


best_logreg = {
    "solver": "lbfgs",
    "C": 1.2542627009984555,
    "max_iter": 1600,
    "random_state": 42,
}


def make_best_logreg(**overrides):
    params = {**best_logreg, **overrides}
    return LogisticRegression(**params)


base_result = run_base_pipeline(
    data_dir="dota-2-hse-ml-1-course-competition-2026",
    model_params=best_logreg,
    submission_path="submission_base_all_features.csv",
)


## 2. Командный чат: покрытие и качество данных

Сначала оцениваю, насколько часто в матчах вообще есть сообщения. Это нужно перед векторизацией: если chat сильно разрежен, он может быть только дополнительным признаком, а не основой модели.


In [ ]:
chat_df = pd.read_csv("dota-2-hse-ml-1-course-competition-2026/game_chat.csv")
print(chat_df.shape)
display(
    chat_df[chat_df["radiant_chat"].notna() | chat_df["dire_chat"].notna()].sample(5)
)

n = len(chat_df)
r_empty = (chat_df["radiant_chat"].fillna("") == "").sum()
d_empty = (chat_df["dire_chat"].fillna("") == "").sum()
both_empty = (
    (chat_df["radiant_chat"].fillna("") == "") & (chat_df["dire_chat"].fillna("") == "")
).sum()

print(f"missing radiant_chat: {r_empty / n:.2%}")
print(f"missing dire_chat:\t   {d_empty / n:.2%}")
print(f"missing both chats:\t   {both_empty / n:.2%}")


**Вывод.** У обеих сторон около `87%` пустых chat полей, а одновременно пустые чаты встречаются примерно в `75.8%` матчей. Chat features могут дать дополнительный сигнал, но coverage ограничен.


### Результаты блока: chat coverage

| Показатель | Значение |
|---|---:|
| Missing `radiant_chat` | 87.07% |
| Missing `dire_chat` | 87.07% |
| Missing both chats | 75.81% |

Интерпретация: чат нельзя считать основным источником сигнала, потому что в большинстве матчей он пустой. Его разумнее использовать как дополнительный sparse-блок поверх baseline.


## 3. Text preprocessing и TF-IDF

Нормализую сообщения: lowercase, tokenization, stop words, лемматизация и фильтрация шума. После этого отдельно строю TF-IDF для Radiant и Dire, чтобы модель могла использовать различия в коммуникации сторон.


In [ ]:
#nltk.download("stopwords", quiet=True)


In [ ]:
morph = pymorphy3.MorphAnalyzer()
ru_stop = set(stopwords.words("russian"))
en_stop = set(stopwords.words("english"))
game_tokens = {"gg", "wp", "ez", "mid", "bot", "top", "afk", "gank", "go"}


@lru_cache(maxsize=200_000)
def lemmatize_ru(token: str):
    return morph.parse(token)[0].normal_form


def preprocessing(text: str):
    if text is None:
        return ""
    text = str(text).lower().replace("|", " ")
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[^a-zа-яё0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = wordpunct_tokenize(text)
    out = []
    for tok in tokens:
        if tok.isdigit():
            continue
        if re.fullmatch(r"[а-яё]+", tok):
            tok = lemmatize_ru(tok)
        if (tok in ru_stop or tok in en_stop) and tok not in game_tokens:
            continue
        if len(tok) <= 1 and tok not in game_tokens:
            continue
        out.append(tok)
    return " ".join(out)


In [ ]:
chat_work = chat_df.copy()
chat_work["radiant_chat"] = chat_work["radiant_chat"].fillna("").map(preprocessing)
chat_work["dire_chat"] = chat_work["dire_chat"].fillna("").map(preprocessing)

train_chat = (
    base_result.X_train[["match_id"]]
    .merge(chat_work, on="match_id", how="left")
    .fillna("")
)
test_chat = (
    base_result.X_test[["match_id"]]
    .merge(chat_work, on="match_id", how="left")
    .fillna("")
)

order = pd.to_datetime(base_result.X_train["date"]).sort_values().index.to_numpy()
X_df = base_result.X_train.iloc[order].reset_index(drop=True)
y_df = base_result.y_train.iloc[order].reset_index(drop=True)
chat_ord = train_chat.iloc[order].reset_index(drop=True)

_, X_hero_train_all, _ = build_hero_matrices(
    base_result.player_df_clean,
    train_match_ids=base_result.X_train["match_id"],
    test_match_ids=base_result.X_test["match_id"],
)
X_hero_ord = X_hero_train_all[order]

ts = TimeSeriesSplit(n_splits=5)
cv_scores = []

for tr_idx, val_idx in ts.split(np.arange(len(X_df))):
    X_tr_df = X_df.iloc[tr_idx]
    X_val_df = X_df.iloc[val_idx]
    y_tr = y_df.iloc[tr_idx]
    y_val = y_df.iloc[val_idx]

    rm_art = fit_region_mmr_artifacts(X_tr_df, y_tr)
    X_rm_tr = transform_region_mmr(X_tr_df, rm_art)
    X_rm_val = transform_region_mmr(X_val_df, rm_art)

    tfidf_r_cv = TfidfVectorizer(
        ngram_range=(1, 2), min_df=5, max_df=0.98, sublinear_tf=True
    )
    tfidf_d_cv = TfidfVectorizer(
        ngram_range=(1, 2), min_df=5, max_df=0.98, sublinear_tf=True
    )

    X_r_tr = tfidf_r_cv.fit_transform(chat_ord.iloc[tr_idx]["radiant_chat"])
    X_d_tr = tfidf_d_cv.fit_transform(chat_ord.iloc[tr_idx]["dire_chat"])
    X_r_val = tfidf_r_cv.transform(chat_ord.iloc[val_idx]["radiant_chat"])
    X_d_val = tfidf_d_cv.transform(chat_ord.iloc[val_idx]["dire_chat"])

    X_fold_tr = hstack([X_rm_tr, X_hero_ord[tr_idx], X_r_tr, X_d_tr], format="csr")
    X_fold_val = hstack([X_rm_val, X_hero_ord[val_idx], X_r_val, X_d_val], format="csr")

    logreg = make_best_logreg()
    logreg.fit(X_fold_tr, y_tr)
    p_val = logreg.predict_proba(X_fold_val)[:, 1]
    cv_scores.append(gini(y_val, p_val))

print(f"base+chat gini: {np.mean(cv_scores):.6f} ± {np.std(cv_scores):.6f}")

rm_art_full = fit_region_mmr_artifacts(base_result.X_train, base_result.y_train)
X_rm_train = transform_region_mmr(base_result.X_train, rm_art_full)
X_rm_test = transform_region_mmr(base_result.X_test, rm_art_full)

_, X_hero_train, X_hero_test = build_hero_matrices(
    base_result.player_df_clean,
    train_match_ids=base_result.X_train["match_id"],
    test_match_ids=base_result.X_test["match_id"],
)

tfidf_r = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.98, sublinear_tf=True)
tfidf_d = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.98, sublinear_tf=True)

X_r_train = tfidf_r.fit_transform(train_chat["radiant_chat"])
X_d_train = tfidf_d.fit_transform(train_chat["dire_chat"])
X_r_test = tfidf_r.transform(test_chat["radiant_chat"])
X_d_test = tfidf_d.transform(test_chat["dire_chat"])

X_chat_train = hstack([X_r_train, X_d_train], format="csr")
X_chat_test = hstack([X_r_test, X_d_test], format="csr")

X_base_train = hstack([X_rm_train, X_hero_train], format="csr")
X_base_test = hstack([X_rm_test, X_hero_test], format="csr")

X_adv_train = hstack([X_base_train, X_chat_train], format="csr")
X_adv_test = hstack([X_base_test, X_chat_test], format="csr")

model_adv = make_best_logreg()
model_adv.fit(X_adv_train, base_result.y_train)

pred_test_adv = model_adv.predict_proba(X_adv_test)[:, 1]
submission_adv = pd.DataFrame(
    {"ID": base_result.X_test["match_id"].values, "Value": pred_test_adv}
)
submission_adv.to_csv("submission_advanced_chat.csv", index=False)
print("saved")


In [ ]:
coef = model_adv.coef_.ravel()
base_dim = X_base_train.shape[1]
chat_coef = coef[base_dim:]

feat_r = np.array([f"R:{w}" for w in tfidf_r.get_feature_names_out()])
feat_d = np.array([f"D:{w}" for w in tfidf_d.get_feature_names_out()])
chat_feats = np.concatenate([feat_r, feat_d])

s = pd.Series(chat_coef, index=chat_feats)
s_r = s[s.index.str.startswith("R:")]
s_d = s[s.index.str.startswith("D:")]

s_r.index = s_r.index.str.replace("R:", "", regex=False)
s_d.index = s_d.index.str.replace("D:", "", regex=False)

k = 80
r_pos = s_r[s_r > 0].sort_values(ascending=False).head(k)
r_neg = s_r[s_r < 0].sort_values().head(k)
d_pos = s_d[s_d > 0].sort_values(ascending=False).head(k)
d_neg = s_d[s_d < 0].sort_values().head(k)


def make_wc(freqs, cmap):
    if len(freqs) == 0:
        freqs = {"empty": 1.0}
    return WordCloud(
        width=900, height=450, background_color="white", colormap=cmap
    ).generate_from_frequencies(freqs)


wc_r_pos = make_wc(r_pos.to_dict(), "Greens")
wc_r_neg = make_wc(np.abs(r_neg).to_dict(), "Reds")
wc_d_pos = make_wc(d_pos.to_dict(), "Greens")
wc_d_neg = make_wc(np.abs(d_neg).to_dict(), "Reds")


fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].imshow(wc_r_pos, interpolation="bilinear")
axes[0, 0].axis("off")
axes[0, 0].set_title("Radiant chat: positive weights")

axes[0, 1].imshow(wc_r_neg, interpolation="bilinear")
axes[0, 1].axis("off")
axes[0, 1].set_title("Radiant chat: negative weights")

axes[1, 0].imshow(wc_d_pos, interpolation="bilinear")
axes[1, 0].axis("off")
axes[1, 0].set_title("Dire chat: positive weights")

axes[1, 1].imshow(wc_d_neg, interpolation="bilinear")
axes[1, 1].axis("off")
axes[1, 1].set_title("Dire chat: negative weights")

plt.tight_layout()
plt.show()


**Вывод.** TF-IDF позволяет добавить признаки командной коммуникации в sparse matrix без ручного словаря. При этом нужно аккуратно обучать vectorizer только на train/fold, чтобы избежать leakage.


### Результаты блока: TF-IDF

Chat features добавлялись как sparse TF-IDF признаки отдельно для Radiant и Dire. Это позволяет линейной модели учитывать различия в коммуникации сторон, но требует аккуратного fold-wise fit vectorizer, чтобы не протекала информация из validation/test.


## 4. Gold/experience advantage

Проверяю временные ряды раннего преимущества Radiant по золоту и опыту. Из них строятся простые агрегаты: последнее значение, среднее, медиана, стандартное отклонение и флаги пропусков.


In [ ]:
adv_df = pd.read_csv("dota-2-hse-ml-1-course-competition-2026/dota_adv.csv")
print(adv_df.shape)
display(adv_df.head(3))


def parse_adv_seq(x):
    if pd.isna(x):
        return np.array([], dtype=float)
    s = str(x).replace("\n", " ").strip().strip("[]")
    return np.fromstring(s, sep=" ").astype(float)


adv_df["gold_seq"] = adv_df["radiant_gold_adv"].map(parse_adv_seq)
adv_df["exp_seq"] = adv_df["radiant_exp_adv"].map(parse_adv_seq)

non_empty_adv = adv_df[
    (adv_df["gold_seq"].map(len) > 0) & (adv_df["exp_seq"].map(len) > 0)
].copy()

sample = non_empty_adv.sample(2, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for i, (_, row) in enumerate(sample.iterrows()):
    g = row["gold_seq"]
    e = row["exp_seq"]

    axes[i, 0].plot(range(len(g)), g)
    axes[i, 0].axhline(0, color="black", linewidth=1, alpha=0.5)
    axes[i, 0].set_title(f"match_id={row['match_id']} gold_adv")

    axes[i, 1].plot(range(len(e)), e)
    axes[i, 1].axhline(0, color="black", linewidth=1, alpha=0.5)
    axes[i, 1].set_title(f"match_id={row['match_id']} exp_adv")

plt.tight_layout()
plt.show()


In [ ]:
adv_df["gold_len"] = adv_df["gold_seq"].map(len)
adv_df["exp_len"] = adv_df["exp_seq"].map(len)

print("empty gold:", (adv_df["gold_len"] == 0).sum())
print("empty exp:", (adv_df["exp_len"] == 0).sum())
print("len mismatch:", (adv_df["gold_len"] != adv_df["exp_len"]).sum())
print("gold_len mode:", adv_df["gold_len"].mode().iloc[0])
print("exp_len mode:", adv_df["exp_len"].mode().iloc[0])

display(adv_df[["gold_len", "exp_len"]].describe())


In [ ]:
def last(arr):
    return arr[-1] if len(arr) > 0 else np.nan


def mean(arr):
    return np.mean(arr) if len(arr) > 0 else np.nan


def median(arr):
    return np.median(arr) if len(arr) > 0 else np.nan


def std(arr):
    return np.std(arr) if len(arr) > 0 else np.nan


adv_features = pd.DataFrame({"match_id": adv_df["match_id"]})
adv_features["adv_missing"] = (
    (adv_df["gold_seq"].map(len) == 0) | (adv_df["exp_seq"].map(len) == 0)
).astype(int)

# gold
adv_features["gold_last"] = adv_df["gold_seq"].map(last)
adv_features["gold_mean"] = adv_df["gold_seq"].map(mean)
adv_features["gold_median"] = adv_df["gold_seq"].map(median)
adv_features["gold_std"] = adv_df["gold_seq"].map(std)

# exp
adv_features["exp_last"] = adv_df["exp_seq"].map(last)
adv_features["exp_mean"] = adv_df["exp_seq"].map(mean)
adv_features["exp_median"] = adv_df["exp_seq"].map(median)
adv_features["exp_std"] = adv_df["exp_seq"].map(std)

adv_cols = [
    "adv_missing",
    "gold_last",
    "gold_mean",
    "gold_median",
    "gold_std",
    "exp_last",
    "exp_mean",
    "exp_median",
    "exp_std",
]


all_ids = pd.Index(base_result.X_train["match_id"]).union(
    base_result.X_test["match_id"]
)
feature_store = pd.DataFrame({"match_id": all_ids})

feature_store = feature_store.drop(
    columns=[c for c in adv_cols if c in feature_store.columns], errors="ignore"
)
feature_store = feature_store.merge(
    adv_features[["match_id"] + adv_cols], on="match_id", how="left"
)

fs_train = base_result.X_train[["match_id"]].merge(
    feature_store, on="match_id", how="left"
)
fs_test = base_result.X_test[["match_id"]].merge(
    feature_store, on="match_id", how="left"
)

fs_train["adv_missing"] = fs_train["adv_missing"].fillna(1)
fs_test["adv_missing"] = fs_test["adv_missing"].fillna(1)

other_adv_cols = [c for c in adv_cols if c != "adv_missing"]
fs_train[other_adv_cols] = fs_train[other_adv_cols].fillna(0.0)
fs_test[other_adv_cols] = fs_test[other_adv_cols].fillna(0.0)

fs_train_raw = fs_train.copy()
fs_test_raw = fs_test.copy()

scaler_adv = StandardScaler()
fs_train_scaled = fs_train.copy()
fs_test_scaled = fs_test.copy()
fs_train_scaled[other_adv_cols] = scaler_adv.fit_transform(fs_train_scaled[other_adv_cols])
fs_test_scaled[other_adv_cols] = scaler_adv.transform(fs_test_scaled[other_adv_cols])

X_stats_train = csr_matrix(fs_train_scaled[adv_cols].to_numpy())
X_stats_test = csr_matrix(fs_test_scaled[adv_cols].to_numpy())


X_final_train = hstack(
    [base_result.X_all_train, X_chat_train, X_stats_train], format="csr"
)
X_final_test = hstack([base_result.X_all_test, X_chat_test, X_stats_test], format="csr")

print("X_stats_train:", X_stats_train.shape)
print("X_final_train:", X_final_train.shape)


**Вывод.** Большая часть матчей не содержит advantage history, поэтому важны не только агрегаты (`last`, `mean`, `std`, etc.), но и missing indicators. Непустые последовательности обычно имеют длину 16.


### Результаты блока: advantage history

| Показатель | Значение |
|---|---:|
| Empty gold sequences | 529,783 |
| Empty exp sequences | 529,783 |
| Mode sequence length | 0 |
| Non-empty typical length | 16 |

Интерпретация: advantage history потенциально сильный источник сигнала, но он разрежен. Поэтому важно добавлять missing indicators, а не только агрегаты по непустым последовательностям.


## 5. Объединение признаков

На этом шаге baseline features, chat TF-IDF и advantage aggregations объединяются в одну sparse matrix. Такой вариант тяжелее baseline, но позволяет проверить вклад дополнительных источников сигнала.


In [ ]:
order = pd.to_datetime(base_result.X_train["date"]).sort_values().index.to_numpy()

X_df = base_result.X_train.iloc[order].reset_index(drop=True)
y_df = base_result.y_train.iloc[order].reset_index(drop=True)

chat_ord = train_chat.iloc[order].reset_index(drop=True)

_, X_hero_train_all, _ = build_hero_matrices(
    base_result.player_df_clean,
    train_match_ids=base_result.X_train["match_id"],
    test_match_ids=base_result.X_test["match_id"],
)
X_hero_ord = X_hero_train_all[order]

fs_train_ord = fs_train_raw.iloc[order].reset_index(drop=True)
adv_cols = [
    "adv_missing",
    "gold_last",
    "gold_mean",
    "gold_median",
    "gold_std",
    "exp_last",
    "exp_mean",
    "exp_median",
    "exp_std",
]
other_adv_cols = [c for c in adv_cols if c != "adv_missing"]

ts = TimeSeriesSplit(n_splits=5)
scores = []

for tr_idx, val_idx in ts.split(np.arange(len(X_df))):
    X_tr_df = X_df.iloc[tr_idx]
    X_val_df = X_df.iloc[val_idx]
    y_tr = y_df.iloc[tr_idx]
    y_val = y_df.iloc[val_idx]

    # region+mmr
    rm_art = fit_region_mmr_artifacts(X_tr_df, y_tr)
    X_rm_tr = transform_region_mmr(X_tr_df, rm_art)
    X_rm_val = transform_region_mmr(X_val_df, rm_art)

    # tfidf
    tfidf_r = TfidfVectorizer(
        ngram_range=(1, 2), min_df=5, max_df=0.98, sublinear_tf=True
    )
    tfidf_d = TfidfVectorizer(
        ngram_range=(1, 2), min_df=5, max_df=0.98, sublinear_tf=True
    )
    X_r_tr = tfidf_r.fit_transform(chat_ord.iloc[tr_idx]["radiant_chat"])
    X_d_tr = tfidf_d.fit_transform(chat_ord.iloc[tr_idx]["dire_chat"])
    X_r_val = tfidf_r.transform(chat_ord.iloc[val_idx]["radiant_chat"])
    X_d_val = tfidf_d.transform(chat_ord.iloc[val_idx]["dire_chat"])

    # scaler
    adv_tr = fs_train_ord.iloc[tr_idx][adv_cols].copy()
    adv_val = fs_train_ord.iloc[val_idx][adv_cols].copy()

    scaler_fold = StandardScaler()
    adv_tr[other_adv_cols] = scaler_fold.fit_transform(
        adv_tr[other_adv_cols].fillna(0.0)
    )
    adv_val[other_adv_cols] = scaler_fold.transform(adv_val[other_adv_cols].fillna(0.0))
    adv_tr["adv_missing"] = adv_tr["adv_missing"].fillna(1)
    adv_val["adv_missing"] = adv_val["adv_missing"].fillna(1)

    X_stats_tr = csr_matrix(adv_tr.to_numpy())
    X_stats_val = csr_matrix(adv_val.to_numpy())

    X_fold_tr = hstack(
        [X_rm_tr, X_hero_ord[tr_idx], X_r_tr, X_d_tr, X_stats_tr], format="csr"
    )
    X_fold_val = hstack(
        [X_rm_val, X_hero_ord[val_idx], X_r_val, X_d_val, X_stats_val], format="csr"
    )

    clf = make_best_logreg()
    clf.fit(X_fold_tr, y_tr)
    p_val = clf.predict_proba(X_fold_val)[:, 1]
    scores.append(gini(y_val, p_val))


print(f"all gini: {np.mean(scores):.6f} ± {np.std(scores):.6f}")
print("folds:", [round(s, 6) for s in scores])


In [ ]:
rm_art = fit_region_mmr_artifacts(base_result.X_train, base_result.y_train)
X_rm_train = transform_region_mmr(base_result.X_train, rm_art)
X_rm_test = transform_region_mmr(base_result.X_test, rm_art)

_, X_hero_train, X_hero_test = build_hero_matrices(
    base_result.player_df_clean,
    train_match_ids=base_result.X_train["match_id"],
    test_match_ids=base_result.X_test["match_id"],
)

tfidf_r = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.98, sublinear_tf=True)
tfidf_d = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.98, sublinear_tf=True)

X_r_train = tfidf_r.fit_transform(train_chat["radiant_chat"])
X_d_train = tfidf_d.fit_transform(train_chat["dire_chat"])
X_r_test = tfidf_r.transform(test_chat["radiant_chat"])
X_d_test = tfidf_d.transform(test_chat["dire_chat"])

adv_cols = [
    "adv_missing",
    "gold_last",
    "gold_mean",
    "gold_median",
    "gold_std",
    "exp_last",
    "exp_mean",
    "exp_median",
    "exp_std",
]
other_adv_cols = [c for c in adv_cols if c != "adv_missing"]

fs_train_full = fs_train_raw.copy()
fs_test_full = fs_test_raw.copy()

scaler_adv_full = StandardScaler()
fs_train_full[other_adv_cols] = scaler_adv_full.fit_transform(
    fs_train_full[other_adv_cols].fillna(0.0)
)
fs_test_full[other_adv_cols] = scaler_adv_full.transform(
    fs_test_full[other_adv_cols].fillna(0.0)
)

X_stats_train = csr_matrix(fs_train_full[adv_cols].to_numpy())
X_stats_test = csr_matrix(fs_test_full[adv_cols].to_numpy())


X_final_train = hstack(
    [X_rm_train, X_hero_train, X_r_train, X_d_train, X_stats_train], format="csr"
)
X_final_test = hstack(
    [X_rm_test, X_hero_test, X_r_test, X_d_test, X_stats_test], format="csr"
)

print("X_final_train:", X_final_train.shape)
print("X_final_test :", X_final_test.shape)

model_final = make_best_logreg()
model_final.fit(X_final_train, base_result.y_train)

pred_test = model_final.predict_proba(X_final_test)[:, 1]
submission_final = pd.DataFrame(
    {
        "ID": base_result.X_test["match_id"].values,
        "Value": pred_test,
    }
)
submission_final.to_csv("submission_final.csv", index=False)
print("saved: submission_final.csv")


## Итоги advanced experiments

- Chat features потенциально полезны, но покрытие слабое: около `75.8%` матчей не имеют сообщений ни у одной стороны.
- Advantage history может нести сильный сигнал о раннем состоянии матча, но большая часть последовательностей пустая, поэтому missing indicators обязательны.
- Расширенная матрица экспериментов имела размер около `641090 x 7795` на train и `59748 x 7795` на test.
- Для публичного репозитория я оставил компактный baseline в `src/dota_ml`, а advanced notebook — как исследовательское продолжение.


### Финальные результаты advanced notebook

| Объект | Shape |
|---|---:|
| Advanced train matrix | 641,090 x 7,795 |
| Advanced test matrix | 59,748 x 7,795 |

Главный итог: chat и advantage features расширяют пространство признаков, но делают pipeline тяжелее. Для публичной воспроизводимой версии я оставил компактный baseline в `src/dota_ml`, а этот notebook показывает направление дальнейшего feature engineering.
